# 11 — Concept analysis: filtro neuroni validi, statistiche e visualizzazioni

Notebook di analisi delle attivazioni SAE sul **test set**, prodotte dal notebook 10.
Il modello è **selezionabile** nella cella di configurazione iniziale (§0): `EXPANSION_FACTOR=8` → latent_dim 6144,
oppure `EXPANSION_FACTOR=16` → latent_dim 12288.

**Cosa fa**:
1. Carica le attivazioni SAE `[N_test, LATENT_DIM]` (memmap float32) e la concept map adattiva del modello selezionato.
2. Filtra i neuroni `valid=True` e restringe la matrice alle sole colonne valide.
3. Calcola la distribuzione del numero di concetti attivi per immagine (istogramma skewed).
4. Calcola la frequenza dei concetti nel dataset e salva una tabella riassuntiva.
5. Mostra 8 immagini campione (varieta' high/low concetti + random) con i top-5 concetti attivi.


## 0. Selezione del modello MSAE

Cella di configurazione: scegli **quale modello** analizzare impostando `EXPANSION_FACTOR`.
I nomi del file di attivazioni (output del nb10) e della concept map (output del nb9) derivano da `LATENT_DIM = 768 × EXPANSION_FACTOR`.
**Deve combaciare** con il valore usato nei notebook 9 e 10.

- `EXPANSION_FACTOR = 8`  → `LATENT_DIM = 6144`  (modello del paper)
- `EXPANSION_FACTOR = 16` → `LATENT_DIM = 12288` (modello a capacità latente raddoppiata)

In [ ]:
# === SELEZIONE DEL MODELLO MSAE ===
# Imposta EXPANSION_FACTOR per scegliere quale modello analizzare:
#   EXPANSION_FACTOR = 8   ->  LATENT_DIM = 6144   (modello del paper)
#   EXPANSION_FACTOR = 16  ->  LATENT_DIM = 12288  (modello a capacità latente raddoppiata)
# Deve combaciare con il modello usato nei notebook 9 (concept map) e 10 (attivazioni).
EXPANSION_FACTOR = 16   # <-- cambia qui: 8 oppure 16

assert EXPANSION_FACTOR in (8, 16), f"EXPANSION_FACTOR deve essere 8 o 16, non {EXPANSION_FACTOR}"
LATENT_DIM = 768 * EXPANSION_FACTOR   # 768*8 = 6144 ; 768*16 = 12288
print(f"Modello MSAE selezionato: expansion_factor={EXPANSION_FACTOR}  ->  latent_dim={LATENT_DIM}")

## 1. Setup Colab — Drive e dipendenze

Monta Google Drive con `force_remount=True` per garantire che eventuali file caricati su Drive dopo un mount precedente siano effettivamente visibili (Colab fa cache aggressiva del mount). Non e' necessario clonare la repo MSAE: questo notebook usa solo numpy/pandas/PIL/matplotlib (preinstallati su Colab).

In [2]:
import os
import sys
import zipfile
import pathlib

from google.colab import drive

# force_remount=True: invalida la cache di Colab e ri-monta Drive. Necessario quando
# file sono stati caricati su Drive dopo un mount precedente: senza remount, Colab
# vede uno snapshot stale e os.path.exists() ritorna False anche se il file esiste.
drive.mount('/content/drive', force_remount=True)

# Decomprime il subset di immagini in locale se non gia' presente (necessario per la cella 6).
# Stesso pattern del notebook 6: estrae output_subset.zip da Drive in /content/tesi/.
SUBSET_LOCAL = pathlib.Path('/content/tesi/output_subset')
ZIP_ON_DRIVE = pathlib.Path('/content/drive/MyDrive/tesi/output_subset.zip')

if not SUBSET_LOCAL.exists() and ZIP_ON_DRIVE.exists():
    print(f'Estrazione di {ZIP_ON_DRIVE.name} -> /content/tesi/ ...')
    SUBSET_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as z:
        z.extractall(SUBSET_LOCAL.parent)
    print('Estrazione completata.')
elif SUBSET_LOCAL.exists():
    print(f"output_subset gia' presente: {SUBSET_LOCAL}")
else:
    print(f'WARNING: ne {SUBSET_LOCAL} ne {ZIP_ON_DRIVE} esistono.')
    print("La cella 6 (visualizzazioni) fallira' nel caricamento immagini.")

Mounted at /content/drive
Estrazione di output_subset.zip -> /content/tesi/ ...
Estrazione completata.


## 2. Configurazione path

Tutte le path configurabili sono in cima. Nessun path hardcoded altrove.

Il file `image_paths_test.csv` mappa `embedding_idx -> path locale Colab dell'immagine`. Se non esiste, viene creato dinamicamente da `clip_image_embedding_manifest_test.csv` (prodotto dal notebook 6) convertendo i path Windows in path Colab — stessa logica `to_colab_path` del notebook 6.

In [ ]:
import pandas as pd

DRIVE_ROOT = '/content/drive/MyDrive/tesi'

# Numero latenti del modello selezionato in §0 (6144 o 12288).
N_LATENTS = LATENT_DIM

# Attivazioni SAE sul test set (memmap float32, shape [N_TEST, LATENT_DIM]). Output del nb10.
N_TEST = 2938
SAE_ACTIVATIONS_PATH = f'{DRIVE_ROOT}/sae_activations_test_{N_TEST}_{LATENT_DIM}.npy'

# Soglia di magnitudo per contare un'attivazione come "concetto attivo".
# >0.0 include il pavimento di ricostruzione (non semantico: paper Sez.4.6,
#      Templeton et al.): p50 attivazioni non-zero = 0.39 -> gia' a >0 si
#      attivano ~7/15 neuroni su ogni immagine in modo quasi uniforme.
# 0.5 mantiene la coda semantica (2.88 concetti/img medi, 3.4% img a 0).
# 1.0 analisi conservativa (1.18 medi, 28.5% a 0).
ACTIVATION_THRESHOLD = 0.5

# Concept map adattiva del modello selezionato (output del notebook 9).
CONCEPT_MAP_PATH = f'{DRIVE_ROOT}/concept_map_adaptive_{LATENT_DIM}.csv'

# Manifest del test set (output del notebook 6).
TEST_MANIFEST_PATH = f'{DRIVE_ROOT}/clip_image_embedding_manifest_test.csv'

# Mapping embedding_idx -> path Colab dell'immagine. Creato al volo se assente.
IMAGE_PATHS_FILE = f'{DRIVE_ROOT}/image_paths_test.csv'

# Root locale Colab dove sono estratte le immagini del subset.
SUBSET_ROOT_LOCAL = '/content/tesi'

OUTPUT_DIR = f'{DRIVE_ROOT}'
CHARTS_DIR = f'{OUTPUT_DIR}/charts'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

# --- Verifica file con diagnostica chiara su miss ---
required_files = {
    'SAE_ACTIVATIONS_PATH': SAE_ACTIVATIONS_PATH,
    'CONCEPT_MAP_PATH'    : CONCEPT_MAP_PATH,
    'TEST_MANIFEST_PATH'  : TEST_MANIFEST_PATH,
}
missing = {k: v for k, v in required_files.items() if not os.path.exists(v)}

if missing:
    print('=== ERRORE: file mancanti ===')
    for k, v in missing.items():
        print(f'  {k}: {v}')
    print(f'Contenuto effettivo di {OUTPUT_DIR}:')
    if os.path.isdir(OUTPUT_DIR):
        for name in sorted(os.listdir(OUTPUT_DIR)):
            full = os.path.join(OUTPUT_DIR, name)
            size = os.path.getsize(full) if os.path.isfile(full) else 0
            kind = 'DIR ' if os.path.isdir(full) else 'FILE'
            print(f'  [{kind}] {name}  ({size/1e6:.1f} MB)' if kind == 'FILE' else f'  [{kind}] {name}')
    else:
        print(f'  (la directory {OUTPUT_DIR} non esiste)')
    raise FileNotFoundError(
        f'File mancanti su Drive: {list(missing.keys())}. '
        'Se i file esistono su Drive ma non sono visibili qui, esegui di nuovo la cella 2 '
        '(force_remount=True) o riavvia il runtime.'
    )

# --- Creazione mapping embedding_idx -> path Colab immagine ---
if not os.path.exists(IMAGE_PATHS_FILE):
    print(f'{IMAGE_PATHS_FILE} non esiste, lo creo da {TEST_MANIFEST_PATH}.')
    df_manifest = pd.read_csv(TEST_MANIFEST_PATH)

    def to_colab_path(win_path: str) -> str:
        p_str = win_path.replace('/', '\\')
        parts = p_str.split('\\')
        try:
            idx = parts.index('output_subset')
            tail = '\\'.join(parts[idx:])
            return os.path.join(SUBSET_ROOT_LOCAL, tail)
        except ValueError:
            return os.path.join(SUBSET_ROOT_LOCAL, parts[-1])

    df_manifest = df_manifest.sort_values('embedding_idx').reset_index(drop=True)
    df_manifest['image_path_colab'] = df_manifest['image_path_dest'].apply(to_colab_path)
    df_paths = df_manifest[['embedding_idx', 'cluster_id', 'category', 'image_path_colab']]
    df_paths.to_csv(IMAGE_PATHS_FILE, index=False)
    print(f'Salvato: {IMAGE_PATHS_FILE}  ({len(df_paths)} righe)')
else:
    print(f"Mapping immagini gia' presente: {IMAGE_PATHS_FILE}")

print(f'Modello (expansion={EXPANSION_FACTOR}, latent={LATENT_DIM})')
print(f'SAE_ACTIVATIONS_PATH : {SAE_ACTIVATIONS_PATH}')
print(f'CONCEPT_MAP_PATH     : {CONCEPT_MAP_PATH}')
print(f'IMAGE_PATHS_FILE     : {IMAGE_PATHS_FILE}')
print(f'ACTIVATION_THRESHOLD : {ACTIVATION_THRESHOLD}')
print(f'CHARTS_DIR           : {CHARTS_DIR}')

## 3. Filtro ai neuroni validi

Carico la concept map e seleziono i neuroni con `valid=True`. Restringo la matrice attivazioni alle sole colonne dei neuroni validi: tutte le analisi a valle lavorano su `[N_TEST, N_valid]`.

In [6]:
import numpy as np

# Attivazioni come memmap read-only (non satura RAM Colab).
activations = np.memmap(SAE_ACTIVATIONS_PATH, dtype=np.float32, mode='r', shape=(N_TEST, N_LATENTS))

concept_map = pd.read_csv(CONCEPT_MAP_PATH)
valid_mask = concept_map['valid'].astype(bool).values
valid_neuron_ids = concept_map.loc[valid_mask, 'neuron_id'].values.astype(int)
valid_concepts = concept_map.loc[valid_mask, 'concept'].values.astype(str)

# Estrae solo le colonne dei neuroni validi -> [N_TEST, N_valid] in RAM (piccolo).
activations_valid = np.asarray(activations[:, valid_neuron_ids], dtype=np.float32)
n_valid = int(valid_mask.sum())

print(f'Neuroni totali         : {N_LATENTS}')
print(f'Neuroni validi         : {n_valid}')
print(f'Matrice filtrata shape : {list(activations_valid.shape)}')
print(f'Concetti validi        : {list(valid_concepts)}')

Neuroni totali         : 6144
Neuroni validi         : 26
Matrice filtrata shape : [2938, 26]
Concetti validi        : [np.str_('instaweather'), np.str_('manatee'), np.str_('donegal'), np.str_('windmill'), np.str_('galway'), np.str_('tampa'), np.str_('conwy'), np.str_('pink'), np.str_('aruba'), np.str_('marseille'), np.str_('bendigo'), np.str_('window'), np.str_('fuji'), np.str_('barns'), np.str_('instaweather'), np.str_('quito'), np.str_('blackandwhitephotography'), np.str_('kyoto'), np.str_('strasbourg'), np.str_('manatee'), np.str_('bed'), np.str_('cafeteria'), np.str_('wexmondays'), np.str_('andon'), np.str_('penang'), np.str_('guangzhou')]


## 3bis. Diagnostica Check 2/3 — neuroni sempre-accesi e soglia di magnitudo

La cella seguente verifica le due cause del sintomo *"sempre gli stessi concetti"*:

- **(3) Neuroni non-discriminanti.** I neuroni "validi" sono scelti solo per similarità decoder↔vocab, **senza** alcun vincolo su come si attivano sulle immagini. Se un neurone è attivo (`>0`) in ~100% delle immagini, non distingue nulla: si accende sempre. Stampiamo la frequenza di attivazione per neurone.

- **(2) Soglia `>0` sul floor di ricostruzione.** In nb10 viene salvato `full_latents` = ReLU **senza** TopK, che include le attivazioni basse "di ricostruzione" che il paper (Sez. 4.6) considera non-semantiche. Contarle con `>0` fa apparire gli stessi concetti ovunque. Alziamo la soglia (0→10) e osserviamo quando i concetti si differenziano.

Il checkpoint ha `soft_cap=0.0` (disattivato), quindi l'ipotesi di saturazione è esclusa: il problema è la soglia, non un tetto sulle attivazioni.


In [ ]:
# === DIAGNOSTICA CHECK 2/3: neuroni "sempre accesi" e soglia di magnitudo ===
# (3) Se un neurone valido e' attivo (>0) in ~100% delle immagini NON e' discriminante:
#     la selezione decoder<->vocab non impone alcuna specificita' per-immagine.
# (2) full_latents (nb10) = ReLU SENZA TopK -> include attivazioni di pura ricostruzione
#     (paper Sez.4.6: le attivazioni basse non hanno significato semantico). Contarle con >0
#     fa apparire gli stessi concetti su ogni immagine. Verifichiamo alzando la soglia.
import numpy as np

acts_v = np.asarray(activations_valid, dtype=np.float32)  # [N_TEST, n_valid]
on = acts_v > 0
freq = on.mean(axis=0) * 100.0
mx = acts_v.max(axis=0)
cnt_on = np.maximum(on.sum(axis=0), 1)
mean_on = (acts_v * on).sum(axis=0) / cnt_on

print('=== (3) Frequenza di attivazione per neurone valido (ordinati per frequenza) ===')
for i in np.argsort(-freq):
    flag = '  <== SEMPRE ACCESO' if freq[i] > 95 else ('  <- raro' if freq[i] < 5 else '')
    print(f'  {str(valid_concepts[i]):>26s} (id {int(valid_neuron_ids[i]):5d}): '
          f'{freq[i]:5.1f}% img | mean_on={mean_on[i]:7.3f} | max={mx[i]:7.3f}{flag}')
n_always = int((freq > 95).sum())
print(f'\nNeuroni sempre-accesi (>95% immagini): {n_always}/{acts_v.shape[1]}'
      f'  -> sono questi a produrre i concetti identici su ogni immagine.')

print('\n=== (2) # concetti attivi per immagine al variare della soglia di magnitudo ===')
print(f'{"soglia":>8}{"media":>9}{"mediana":>9}{"%img=0":>9}')
for thr in [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]:
    cpi = (acts_v > thr).sum(axis=1)
    print(f'{thr:>8.1f}{cpi.mean():>9.2f}{np.median(cpi):>9.1f}{100.0*(cpi==0).mean():>8.1f}%')

nz = acts_v[on]
if nz.size:
    print('\n=== Magnitudo attivazioni non-zero (neuroni validi) ===')
    print(f'  p50={np.percentile(nz,50):.3f}  p90={np.percentile(nz,90):.3f}  '
          f'p99={np.percentile(nz,99):.3f}  max={nz.max():.3f}')
print('  NB: checkpoint con soft_cap=0.0 -> nessun soft-cap (attivazioni illimitate): la')
print('      saturazione e\' esclusa, il problema e\' la soglia >0 sul pavimento di ricostruzione.')
print('\nLettura: se gia\' a soglia 1-2 la maggior parte dei concetti sparisce, il "sempre uguali"')
print('  era il floor di ricostruzione. Soluzioni: usare l\'output TopK sparso (latents, non')
print('  full_latents, in nb10) oppure analizzare solo la coda alta di magnitudo.')


## 4. Distribuzione del numero di concetti per immagine

Per ogni immagine del test set conto quanti neuroni validi hanno attivazione `> 0`. Mi aspetto una distribuzione **skewed**: la sparsity media del MSAE TopK-64 fa si' che ogni immagine abbia attivi solo pochi neuroni, e tra questi soltanto una frazione e' tra i validi. Tipicamente: molte immagini con 0–2 concetti, coda lunga verso valori piu' alti.

In [ ]:
import matplotlib.pyplot as plt

# Conta i concetti attivi con la soglia configurabile (non >0 che include il
# pavimento di ricostruzione). Check 2/3 ha mostrato: p50 attivazioni = 0.39,
# a soglia 0.5 si ottengono 2.88 concetti/img medi con 3.4% immagini a zero.
concepts_per_image = (activations_valid > ACTIVATION_THRESHOLD).sum(axis=1).astype(int)

mean_concepts = float(concepts_per_image.mean())
median_concepts = float(np.median(concepts_per_image))
pct_zero = 100.0 * float((concepts_per_image == 0).mean())
pct_gt5 = 100.0 * float((concepts_per_image > 5).mean())

print(f'Soglia attivazione                     : {ACTIVATION_THRESHOLD}')
print(f'Media concetti attivi per immagine     : {mean_concepts:.2f}')
print(f'Mediana concetti attivi per immagine   : {median_concepts:.1f}')
print(f'% immagini con 0 concetti attivi       : {pct_zero:.2f}%')
print(f'% immagini con > 5 concetti attivi     : {pct_gt5:.2f}%')
print(f'Min / Max concetti per immagine        : {concepts_per_image.min()} / {concepts_per_image.max()}')

max_c = int(concepts_per_image.max())
bins = np.arange(0, max_c + 2) - 0.5

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(concepts_per_image, bins=bins, color='steelblue', edgecolor='black', alpha=0.9)
ax.axvline(mean_concepts, color='red', linestyle='--', label=f'media = {mean_concepts:.2f}')
ax.axvline(median_concepts, color='darkorange', linestyle=':', label=f'mediana = {median_concepts:.1f}')
ax.set_xlabel('Numero di concetti attivi per immagine')
ax.set_ylabel('Numero di immagini')
ax.set_title(f'Distribuzione concetti attivi per immagine (soglia>{ACTIVATION_THRESHOLD})')
ax.legend()
fig.tight_layout()

out_path = os.path.join(CHARTS_DIR, 'plot_concepts_per_image.png')
fig.savefig(out_path, dpi=150)
plt.show()
print(f'\nGrafico salvato: {out_path}')


## 5. Frequenza dei concetti nel dataset

Per ogni concetto valido conto:
- in **quante immagini** si attiva (attivazione `> 0`),
- la **percentuale** di immagini coperte,
- la **magnitudo media** dell'attivazione quando il concetto e' attivo (informa su quanto "forte" e' il concetto, non solo su quanto e' frequente).

Salvo il bar chart dei top-20 in `CHARTS_DIR` e la tabella completa ordinata per frequenza in `OUTPUT_DIR/concept_stats.csv`.

In [ ]:
active_mask = activations_valid > ACTIVATION_THRESHOLD  # [N_TEST, N_valid]
n_images_active = active_mask.sum(axis=0).astype(int)
pct_images_active = 100.0 * n_images_active / N_TEST

# Mean activation when active: solo i valori sopra soglia.
sum_when_active = (activations_valid * active_mask).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    mean_activation_when_active = np.where(
        n_images_active > 0,
        sum_when_active / np.maximum(n_images_active, 1),
        0.0,
    ).astype(np.float32)

concept_stats = pd.DataFrame({
    'concept': valid_concepts,
    'neuron_id': valid_neuron_ids,
    'n_images_active': n_images_active,
    'pct_images_active': pct_images_active.round(2),
    'mean_activation_when_active': mean_activation_when_active.round(4),
}).sort_values('n_images_active', ascending=False).reset_index(drop=True)

# Nome con LATENT_DIM: separa le statistiche dei due modelli (6144 vs 12288).
stats_path = os.path.join(OUTPUT_DIR, f'concept_stats_{LATENT_DIM}.csv')
concept_stats.to_csv(stats_path, index=False)
print(f'Tabella salvata: {stats_path}  (soglia={ACTIVATION_THRESHOLD})')
print(f'\nPrime 10 righe:')
print(concept_stats.head(10).to_string(index=False))

top20 = concept_stats.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top20['concept'], top20['n_images_active'], color='darkorange', edgecolor='black')
ax.set_xlabel('Numero di immagini in cui il concetto e\' attivo')
ax.set_ylabel('Concetto')
ax.set_title(f'Top 20 concetti piu\' frequenti nel test set (soglia>{ACTIVATION_THRESHOLD})')
for i, (n, c) in enumerate(zip(top20['n_images_active'], top20['concept'])):
    ax.text(n, i, f' {n}', va='center', fontsize=8)
fig.tight_layout()

out_path = os.path.join(CHARTS_DIR, 'plot_top20_concept_frequency.png')
fig.savefig(out_path, dpi=150)
plt.show()
print(f'\nGrafico salvato: {out_path}')


## 6. Visualizzazione immagini con concetti

Mostro 8 immagini campione del test set, scelte per coprire varieta':
- 2 immagini con il **numero massimo** di concetti attivi (cosa attiva molto il MSAE),
- 2 con il **numero minimo** (escludendo eventuali zero — cosa risulta sotto-rappresentato),
- 4 immagini **casuali** (per evitare il bias verso gli estremi).

Per ognuna, il pannello mostra:
- a **sinistra**, l'immagine originale,
- a **destra**, un bar chart orizzontale con i **top-5 concetti** ordinati per **magnitudo di attivazione** decrescente.

**Come leggere la magnitudo**: l'asse x riporta il valore raw dell'attivazione del neurone associato al concetto. Valori piu' alti indicano che la feature del MSAE corrispondente a quel concetto e' fortemente attivata da quell'immagine. La scala non e' normalizzata tra immagini: confrontare le barre tra pannelli diversi e' indicativo, non rigoroso. Il titolo del pannello riporta il numero totale di concetti attivi nell'immagine.

In [ ]:
from PIL import Image

df_paths = pd.read_csv(IMAGE_PATHS_FILE).sort_values('embedding_idx').reset_index(drop=True)
assert len(df_paths) == N_TEST, f'Mismatch: {len(df_paths)} righe vs N_TEST={N_TEST}'

rng = np.random.default_rng()

available = np.arange(N_TEST)
random_idx = rng.choice(available, size=8, replace=False).tolist()

selected = [
    ('casuale', random_idx[0]),
    ('casuale', random_idx[1]),
    ('casuale', random_idx[2]),
    ('casuale', random_idx[3]),
    ('casuale', random_idx[4]),
    ('casuale', random_idx[5]),
    ('casuale', random_idx[6]),
    ('casuale', random_idx[7]),
]

# Layout: griglia 4x2 di pannelli; ogni pannello = img | bar chart affiancati,
# realizzato con subplot_mosaic (4 righe x 4 colonne effettive, 2 pannelli per riga).
mosaic = []
for r in range(4):
    mosaic.append([f'img_{r*2}', f'bar_{r*2}', f'img_{r*2+1}', f'bar_{r*2+1}'])

fig, axd = plt.subplot_mosaic(
    mosaic,
    figsize=(18, 20),
    gridspec_kw={'width_ratios': [1.0, 1.0, 1.0, 1.0]},
)

for panel_i, (tag, img_idx) in enumerate(selected):
    ax_img = axd[f'img_{panel_i}']
    ax_bar = axd[f'bar_{panel_i}']

    img_path = df_paths.iloc[int(img_idx)]['image_path_colab']
    category = df_paths.iloc[int(img_idx)]['category']
    n_active = int(concepts_per_image[int(img_idx)])

    try:
        img = Image.open(img_path).convert('RGB')
        ax_img.imshow(img)
    except Exception as e:
        ax_img.text(
            0.5, 0.5, f'Errore caricamento\n{e}',
            ha='center', va='center',
            transform=ax_img.transAxes, fontsize=8, color='red',
        )
    ax_img.set_xticks([])
    ax_img.set_yticks([])
    ax_img.set_title(
        f'[{tag}] idx={int(img_idx)} - {category}\n# concetti attivi: {n_active}',
        fontsize=10,
    )

    row_acts = activations_valid[int(img_idx)]
    top_n = min(5, n_active) if n_active > 0 else 0
    if top_n > 0:
        top_neuron_local = np.argsort(-row_acts)[:top_n]
        top_acts = row_acts[top_neuron_local]
        top_names = valid_concepts[top_neuron_local]
        ax_bar.barh(
            np.arange(top_n)[::-1],
            top_acts[::-1],
            color='darkorange',
            edgecolor='black',
        )
        ax_bar.set_yticks(np.arange(top_n)[::-1])
        ax_bar.set_yticklabels(top_names[::-1], fontsize=9)
        ax_bar.set_xlabel('magnitudo attivazione')
        ax_bar.set_title(f'Top {top_n} concetti', fontsize=10)
        for i, v in enumerate(top_acts[::-1]):
            ax_bar.text(v, i, f' {v:.2f}', va='center', fontsize=8)
    else:
        ax_bar.text(
            0.5, 0.5, 'Nessun concetto attivo',
            ha='center', va='center',
            transform=ax_bar.transAxes, fontsize=10,
        )
        ax_bar.set_xticks([])
        ax_bar.set_yticks([])
        ax_bar.set_title('Top concetti', fontsize=10)

fig.suptitle('Immagini campione del test set con i top-5 concetti attivi', fontsize=14, y=1.00)
fig.tight_layout()

out_path = os.path.join(CHARTS_DIR, 'plot_image_concepts_grid.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico salvato: {out_path}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/tesi/outputs/image_paths_test.csv'